<a href="https://colab.research.google.com/github/elnazshokrollahi/My-Agentic-Journey/blob/main/RAG_Pipeline/RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain-openai langchain-community langchain-text-splitters langchain-core chromadb python-dotenv

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_APIKEY')
print("Key loaded ✅")

Key loaded ✅


In [6]:
# -*- coding: utf-8 -*-
"""
Hour 3: Building the Complete RAG Pipeline Demo
================================================

This demo teaches:
1. Complete RAG architecture: retrieve → inject → generate
2. LangChain components (retrievers, prompts, chains)
3. Anti-hallucination strategies
4. Building a production-ready Q&A system

LEARNING RESOURCES:
- RAG Paper (Lewis et al.): https://arxiv.org/abs/2005.11401
- LangChain Documentation: https://python.langchain.com/docs/get_started/introduction
- LCEL Guide: https://python.langchain.com/docs/expression_language/
- Prompt Engineering: https://platform.openai.com/docs/guides/prompt-engineering
- Chroma Vector DB: https://docs.trychroma.com/
"""

import json
import os
# LangChain is a framework for building LLM applications
# Reference: https://python.langchain.com/docs/get_started/introduction
from langchain_openai import OpenAIEmbeddings, ChatOpenAI  # OpenAI integrations
from langchain_community.vectorstores import Chroma  # Vector database for similarity search
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Smart text chunking
from langchain_core.documents import Document  # Document abstraction
from langchain_core.prompts import ChatPromptTemplate  # Prompt templates
from langchain_core.output_parsers import StrOutputParser  # Parse LLM output
from langchain_core.runnables import RunnablePassthrough  # Pass data through pipeline

# Load environment variables (API keys, model names)
from dotenv import load_dotenv
load_dotenv()

print("="*80)
print("HOUR 3: BUILDING THE RAG PIPELINE")
print("="*80)

# ============================================================================
# PART 1: Data Ingestion & Vector Store Setup
# ============================================================================
print("\n" + "="*80)
print("PART 1: Data Ingestion Pipeline")
print("="*80)

# Load tickets
with open('/content/drive/MyDrive/synthetic_tickets.json', 'r') as f:
    tickets = json.load(f)
print(f"✓ Loaded {len(tickets)} support tickets")

# Convert to LangChain Document objects
# Documents are the core abstraction in LangChain - they combine content with metadata
# Reference: https://python.langchain.com/docs/modules/data_connection/document_loaders/
documents = []
for ticket in tickets:
    # Create rich document with all context
    # TIP: Structure your content logically - LLMs understand formatted text better
    content = f"""
Ticket ID: {ticket['ticket_id']}
Title: {ticket['title']}
Category: {ticket['category']}
Priority: {ticket['priority']}
Date: {ticket['created_date']} to {ticket['resolved_date']}

Problem Description:
{ticket['description']}

Resolution:
{ticket['resolution']}
    """.strip()

    # Create Document with metadata
    # Metadata is crucial for filtering, citation, and source tracking
    # Best practice: Include all information you might want to filter or display later
    doc = Document(
        page_content=content,  # The actual text content
        metadata={  # Structured data about the document
            'ticket_id': ticket['ticket_id'],
            'title': ticket['title'],
            'category': ticket['category'],
            'priority': ticket['priority'],
            'source': f"Ticket {ticket['ticket_id']}"
        }
    )
    documents.append(doc)

print(f"✓ Created {len(documents)} documents with metadata")

# Initialize OpenAI embeddings
# Embeddings convert text into vectors for semantic search
# Reference: https://platform.openai.com/docs/guides/embeddings
print("\nInitializing OpenAI embedding model...")
embeddings = OpenAIEmbeddings(
    model=os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')
)
print("✓ OpenAI embedding model ready")

# Build vector store using Chroma
# Chroma is an open-source vector database optimized for AI applications
# It stores embeddings and enables fast similarity search
# Reference: https://docs.trychroma.com/
print("\nBuilding Chroma vector store...")
vector_store = Chroma.from_documents(
    documents=documents,  # Our support ticket documents
    embedding=embeddings,  # Embedding function to use
    collection_name="supportdesk_rag",  # Name for this collection
    persist_directory="./rag_vectorstore"  # Where to save the database
)
print("✓ Vector store created and persisted")

# ============================================================================
# PART 2: Create Retriever
# ============================================================================
print("\n" + "="*80)
print("PART 2: Setting Up Retriever")
print("="*80)

# Create a retriever from the vector store
# Retrievers are the interface for querying the vector store
# Reference: https://python.langchain.com/docs/modules/data_connection/retrievers/
retriever = vector_store.as_retriever(
    search_type="similarity",  # Use cosine similarity for ranking
    search_kwargs={"k": 3}  # Retrieve top-3 most similar documents
    # Other options:
    # - "mmr" (Maximal Marginal Relevance): Balances relevance with diversity
    # - "similarity_score_threshold": Only return docs above a score threshold
)

print("✓ Retriever configured:")
print(f"  - Search type: similarity")
print(f"  - Top-K results: 3")
print("\nTIP: k=3-5 is usually optimal. Too few → missing context, too many → noise")

# Test retriever
test_query = "Users can't log in after changing passwords"
print(f"\nTest query: '{test_query}'")
retrieved_docs = retriever.invoke(test_query)

print(f"\nRetrieved {len(retrieved_docs)} documents:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n#{i} - {doc.metadata['ticket_id']}: {doc.metadata['title']}")
    print(f"  Category: {doc.metadata['category']}")

# ============================================================================
# PART 3: Create Prompt Template with Anti-Hallucination Rules
# ============================================================================
print("\n" + "="*80)
print("PART 3: Prompt Engineering for RAG")
print("="*80)

# Define strict grounding prompt
# Prompt engineering is CRUCIAL for RAG - it tells the LLM how to use the context
# Reference: https://platform.openai.com/docs/guides/prompt-engineering
# Key principles:
# 1. Be explicit about using ONLY the provided context
# 2. Define what to do when information is missing
# 3. Request citations for transparency and verification
# 4. Set the role/persona for appropriate tone
prompt_template = """You are SupportDesk AI, a technical support assistant that helps engineers troubleshoot issues using historical support ticket data.

CRITICAL RULES:
1. Answer ONLY using information from the provided context
2. If the answer is not in the context, say "I don't have enough information in the ticket history to answer that question."
3. DO NOT make up information or use external knowledge
4. Always cite the ticket ID when referencing information
5. If multiple tickets are relevant, mention all of them

Context from support tickets:
{context}

Question: {question}

Helpful Answer (with ticket citations):"""

# Convert string template to ChatPromptTemplate
# This creates a reusable template with variable placeholders
# Reference: https://python.langchain.com/docs/modules/model_io/prompts/
PROMPT = ChatPromptTemplate.from_template(prompt_template)

print("✓ Prompt template created with anti-hallucination rules:")
print("\n" + "-"*80)
print(prompt_template)
print("-"*80)

# ============================================================================
# PART 4: Initialize LLM
# ============================================================================
print("\n" + "="*80)
print("PART 4: Initializing Language Model")
print("="*80)

# Check if OpenAI key is available
if os.getenv("OPENAI_API_KEY"):
    print("✓ OpenAI API key found")
    # Initialize ChatOpenAI for generation
    # Reference: https://python.langchain.com/docs/integrations/chat/openai
    llm = ChatOpenAI(
        model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
        temperature=0,  # Temperature controls randomness (0 = deterministic, 2 = very creative)
        # For RAG, use temperature=0 to ensure consistent, factual responses
        # Reference: https://platform.openai.com/docs/guides/text-generation/how-should-i-set-the-temperature-parameter
        timeout=120,  # Increase timeout for slower connections
        max_retries=3,  # Retry on transient failures
    )
    print(f"✓ Using {os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini')}")
else:
    print("⚠ OpenAI API key not found!")
    print("  Please set OPENAI_API_KEY environment variable")
    print("  Or use Ollama: ollama pull llama2")
    print("\nFor this demo, we'll show the prompt without generating answers.")
    llm = None

# ============================================================================
# PART 5: Build RAG Chain using LCEL (LangChain Expression Language)
# ============================================================================
print("\n" + "="*80)
print("PART 5: Assembling RAG Chain")
print("="*80)

# Helper function to format retrieved documents
# This concatenates all retrieved document contents into a single context string
def format_docs(docs):
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

if llm:
    # Build RAG chain using LCEL (LangChain Expression Language)
    # LCEL allows you to chain components using the | operator (like Unix pipes)
    # Reference: https://python.langchain.com/docs/expression_language/
    #
    # This chain does:
    # 1. Takes a question (string input)
    # 2. Retriever gets relevant docs, format_docs combines them
    # 3. PROMPT fills in {context} and {question} variables
    # 4. LLM generates answer based on filled prompt
    # 5. StrOutputParser extracts the string response
    #
    # The dict {"context": ..., "question": ...} creates the input for the prompt
    qa_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | PROMPT
        | llm
        | StrOutputParser()
    )
    print("✓ RAG chain assembled:")
    print("  Retriever → Context Injection → LLM → Answer")
    print("\nThis is the complete RAG pipeline! Query in → Answer out")
else:
    qa_chain = None
    print("⚠ LLM not available, showing architecture only")

# ============================================================================
# PART 6: Test the RAG System
# ============================================================================
print("\n" + "="*80)
print("PART 6: Testing the RAG System")
print("="*80)

test_queries = [
    "How do I fix authentication failures after password reset?",
    "What causes database connection timeouts?",
    "Why are emails not being delivered?",
    "How do I make the perfect pizza?"  # Should refuse to answer!
]

for query in test_queries:
    print("\n" + "="*80)
    print(f"QUERY: {query}")
    print("="*80)

    # Show retrieved context
    docs = retriever.invoke(query)
    print(f"\nRetrieved {len(docs)} relevant tickets:")
    for i, doc in enumerate(docs, 1):
        print(f"\n  [{i}] {doc.metadata['ticket_id']}: {doc.metadata['title']}")

    if qa_chain:
        # Generate answer
        print("\nGenerating answer...")
        result = qa_chain.invoke(query)

        print("\n" + "-"*80)
        print("ANSWER:")
        print("-"*80)
        print(result)

        print("\n" + "-"*80)
        print("SOURCE DOCUMENTS:")
        print("-"*80)
        for i, doc in enumerate(docs, 1):
            print(f"{i}. {doc.metadata['source']}")
    else:
        print("\n(LLM not configured - would generate answer here)")

# ============================================================================
# PART 7: Advanced - Custom Chain with Validation
# ============================================================================
print("\n" + "="*80)
print("PART 7: Enhanced RAG with Answer Validation")
print("="*80)

def rag_with_validation(query, retriever, llm, min_similarity_score=0.7):
    """
    RAG pipeline with additional validation and fallback
    """
    # Retrieve documents with scores
    docs_with_scores = vector_store.similarity_search_with_score(query, k=3)

    print(f"\nQuery: {query}")
    print(f"\nRetrieval scores:")
    for doc, score in docs_with_scores:
        print(f"  - {doc.metadata['ticket_id']}: {score:.4f}")

    # Check if best match is good enough
    best_score = docs_with_scores[0][1]

    if best_score > min_similarity_score:
        print(f"\n⚠ Best match score ({best_score:.4f}) below threshold ({min_similarity_score})")
        return "I don't have enough relevant information in the ticket history to answer that question confidently."

    # If good matches, proceed with RAG
    docs = [doc for doc, score in docs_with_scores]
    context = "\n\n---\n\n".join([doc.page_content for doc in docs])

    prompt = f"""{prompt_template.replace('{context}', context).replace('{question}', query)}"""

    if llm:
        response = llm.predict(prompt)
        return response
    else:
        return "(LLM not configured)"

print("\nTesting validation logic:")
print("\n1. Relevant query (should answer):")
rag_with_validation(
    "How to fix database connection timeouts?",
    retriever,
    llm,
    min_similarity_score=0.7
)

print("\n2. Irrelevant query (should refuse):")
rag_with_validation(
    "What is the capital of France?",
    retriever,
    llm,
    min_similarity_score=0.7
)

# ============================================================================
# PART 8: Interactive Demo
# ============================================================================
print("\n" + "="*80)
print("PART 8: Interactive SupportDesk Assistant")
print("="*80)

if qa_chain:
    print("\nSupportDesk RAG Assistant Ready!")
    print("Ask questions about support ticket history.")
    print("Type 'quit' to exit.\n")

    while True:
        user_query = input("You: ").strip()

        if user_query.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break

        if not user_query:
            continue

        print("\nAssistant: ", end="")
        answer = qa_chain.invoke(user_query)
        print(answer)

        docs = retriever.invoke(user_query)
        print(f"\n📎 Sources: {', '.join([doc.metadata['ticket_id'] for doc in docs])}")
        print()
else:
    print("\n⚠ Interactive mode requires OpenAI API key")
    print("Set OPENAI_API_KEY to try the interactive assistant!")

print("\n" + "="*80)
print("DEMO COMPLETE!")
print("="*80)
print("\nKey Takeaways:")
print("1. RAG pipeline: Retrieve → Inject Context → Generate")
print("2. Strict prompt engineering prevents hallucinations")
print("3. Always return source documents for verification")
print("4. Implement fallbacks for low-confidence matches")
print("5. Temperature=0 for deterministic, grounded answers")
print("\nNext: Hour 4 - Evaluation & Metrics")

HOUR 3: BUILDING THE RAG PIPELINE

PART 1: Data Ingestion Pipeline
✓ Loaded 20 support tickets
✓ Created 20 documents with metadata

Initializing OpenAI embedding model...
✓ OpenAI embedding model ready

Building Chroma vector store...
✓ Vector store created and persisted

PART 2: Setting Up Retriever
✓ Retriever configured:
  - Search type: similarity
  - Top-K results: 3

TIP: k=3-5 is usually optimal. Too few → missing context, too many → noise

Test query: 'Users can't log in after changing passwords'

Retrieved 3 documents:

#1 - TICK-001: Users unable to log in after password reset
  Category: Authentication

#2 - TICK-011: SSO authentication broken after upgrade
  Category: Authentication

#3 - TICK-016: Two-factor authentication codes not working
  Category: Authentication

PART 3: Prompt Engineering for RAG
✓ Prompt template created with anti-hallucination rules:

--------------------------------------------------------------------------------
You are SupportDesk AI, a technic

In [7]:
retriever_mmr = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10}
)

In [8]:
queries = [
    "Payment processing failures",
    "Mobile app crashes",
    "Slow dashboard loading",
]

for q in queries:
    print("=" * 70)
    print(f"QUERY: {q}")
    print("=" * 70)

    # --- Knob 1: how k changes what's retrieved ---
    for k in [1, 3, 5]:
        r = vector_store.as_retriever(search_kwargs={"k": k})
        ids = [d.metadata["ticket_id"] for d in r.invoke(q)]
        print(f"  k={k:<2} → {ids}")

    # --- Knob 2: similarity vs MMR (both k=3) ---
    sim = vector_store.as_retriever(search_kwargs={"k": 3})
    sim_ids = [d.metadata["ticket_id"] for d in sim.invoke(q)]
    mmr_ids = [d.metadata["ticket_id"] for d in retriever_mmr.invoke(q)]
    print(f"  similarity → {sim_ids}")
    print(f"  mmr        → {mmr_ids}")
    print()

QUERY: Payment processing failures
  k=1  → ['TICK-003']
  k=3  → ['TICK-003', 'TICK-013', 'TICK-008']
  k=5  → ['TICK-003', 'TICK-013', 'TICK-008', 'TICK-009', 'TICK-016']
  similarity → ['TICK-003', 'TICK-013', 'TICK-008']
  mmr        → ['TICK-003', 'TICK-013', 'TICK-019']

QUERY: Mobile app crashes
  k=1  → ['TICK-004']
  k=3  → ['TICK-004', 'TICK-018', 'TICK-010']
  k=5  → ['TICK-004', 'TICK-018', 'TICK-010', 'TICK-016', 'TICK-005']
  similarity → ['TICK-004', 'TICK-018', 'TICK-010']
  mmr        → ['TICK-004', 'TICK-010', 'TICK-016']

QUERY: Slow dashboard loading
  k=1  → ['TICK-010']
  k=3  → ['TICK-010', 'TICK-018', 'TICK-014']
  k=5  → ['TICK-010', 'TICK-018', 'TICK-014', 'TICK-002', 'TICK-017']
  similarity → ['TICK-010', 'TICK-018', 'TICK-014']
  mmr        → ['TICK-010', 'TICK-014', 'TICK-013']



In [9]:
citation_prompt = """Answer the question using the context. Include inline citations [TICK-XXX] after each fact.

Example format:
"Database connection timeouts occur when the pool is undersized [TICK-002]. Increase max_connections [TICK-002]."

Context:
{context}

Question: {question}

Answer with inline citations:"""

CITATION_PROMPT = ChatPromptTemplate.from_template(citation_prompt)

# rebuild the chain with the new prompt (everything else identical to demo.py's chain)
citation_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | CITATION_PROMPT
    | llm
    | StrOutputParser()
)

print(citation_chain.invoke("How do I fix authentication failures after password reset?"))

To fix authentication failures after a password reset, you need to address the issue of session tokens not being invalidated after the password hash algorithm was updated. The solution involves clearing all active sessions and forcing re-authentication for users who have reset their passwords. Additionally, implementing automatic session cleanup on password change will help prevent similar issues in the future [TICK-001].


In [10]:
def smart_rag(query, vector_store, qa_chain):
    """RAG with intelligent fallbacks"""
    docs_with_scores = vector_store.similarity_search_with_score(query, k=3)

    if not docs_with_scores:
        return "No relevant tickets found."

    best_distance = docs_with_scores[0][1]

    if best_distance < 1.2:        # very relevant
        return qa_chain.invoke(query)
    elif best_distance < 1.6:      # somewhat relevant
        ticket_id = docs_with_scores[0][0].metadata['ticket_id']
        return f"Found possibly relevant ticket ({ticket_id}), but confidence is moderate."
    else:                          # not relevant
        return "I don't have relevant ticket history for this question."


tests = [
    ("High confidence", "authentication problems"),
    ("Medium confidence", "system performance"),
    ("Low confidence", "how to bake cookies"),
]

for label, q in tests:
    print("=" * 70)
    print(f"{label}: {q}")
    # show the actual distance so you can SEE why each branch fires
    dist = vector_store.similarity_search_with_score(q, k=3)[0][1]
    print(f"best distance = {dist:.3f}")
    print("→", smart_rag(q, vector_store, qa_chain))
    print()

High confidence: authentication problems
best distance = 1.107
→ There are several authentication problems documented in the support tickets:

1. **Ticket ID: TICK-001** - Users were unable to log in after a password reset, receiving an 'Invalid credentials' error. This issue arose after a recent security patch deployment, where the password hash algorithm was updated but session tokens weren't invalidated. The resolution involved clearing all active sessions and forcing re-authentication, along with implementing automatic session cleanup on password change.

2. **Ticket ID: TICK-016** - Users experienced issues with two-factor authentication (TOTP codes) not working, as the codes were shown as invalid even when entered immediately after generation. This was due to server time drifting by 2 minutes because of a misconfigured NTP. The resolution included configuring NTP synchronization, verifying time accuracy, and increasing the TOTP acceptance window.

3. **Ticket ID: TICK-011** - The

In [11]:
import time

query = "How do I fix database timeouts?"

# ---------- STRATEGY 1: STUFF (all docs, one call) ----------
stuff_prompt = ChatPromptTemplate.from_template(
    "Answer using context:\n\nContext: {context}\n\nQuestion: {question}"
)
stuff_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | stuff_prompt | llm | StrOutputParser()
)

t0 = time.time()
stuff_answer = stuff_chain.invoke(query)
stuff_time = time.time() - t0

# ---------- STRATEGY 2: MAP-REDUCE (each doc, then combine) ----------
# MAP prompt: summarize ONE doc with respect to the question
single_doc_prompt = ChatPromptTemplate.from_template(
    "Given this ticket, what does it say about the question? "
    "If nothing, say 'Not relevant.'\n\nTicket: {doc}\n\nQuestion: " + query
)
# REDUCE prompt: merge the per-doc notes into one answer
combine_prompt = ChatPromptTemplate.from_template(
    "Combine these notes into one clear answer with ticket citations.\n\nNotes:\n{summaries}"
)

t0 = time.time()
docs = retriever.invoke(query)
map_chain = single_doc_prompt | llm | StrOutputParser()
individual_answers = [map_chain.invoke({"doc": d.page_content}) for d in docs]   # MAP
combine_chain = combine_prompt | llm | StrOutputParser()
mapreduce_answer = combine_chain.invoke({"summaries": "\n".join(individual_answers)})  # REDUCE
mapreduce_time = time.time() - t0

# ---------- COMPARE ----------
print(f"STUFF        → {len(docs)} docs, 1 API call,  {stuff_time:.2f}s")
print(f"MAP-REDUCE   → {len(docs)} docs, {len(docs)+1} API calls, {mapreduce_time:.2f}s")
print("\n--- STUFF ANSWER ---\n", stuff_answer)
print("\n--- MAP-REDUCE ANSWER ---\n", mapreduce_answer)

STUFF        → 3 docs, 1 API call,  2.50s
MAP-REDUCE   → 3 docs, 4 API calls, 3.30s

--- STUFF ANSWER ---
 To fix database timeouts, you can take the following steps based on the resolution from Ticket ID: TICK-002:

1. **Increase Connection Pool Size**: If your application is experiencing 'connection pool exhausted' errors, it may indicate that the current maximum number of database connections is too low for peak load. Consider increasing the `max_connections` setting in your database configuration to accommodate more simultaneous connections.

2. **Monitor Connection Pooling**: Implement monitoring and alerts for your connection pool to track usage and identify potential issues before they affect application performance.

3. **Optimize Long-Running Queries**: Analyze your database queries to identify any that are taking too long to execute. Optimize these queries by rewriting them for efficiency, adding appropriate indexes, or breaking them into smaller, more manageable queries.

4.

In [12]:
sample = vector_store.similarity_search("system problem", k=20)
print("Categories:", set(d.metadata['category'] for d in sample))
print("Priorities:", set(d.metadata['priority'] for d in sample))

Categories: {'Search', 'Email', 'Payment', 'Database', 'API', 'Mobile', 'Integration', 'Media Processing', 'Real-time', 'Performance', 'Export', 'Reporting', 'File Upload', 'User Management', 'Authentication'}
Priorities: {'High', 'Critical', 'Medium'}


In [13]:
query = "system problem"

def show(label, docs):
    print(f"\n{label}")
    for d in docs:
        print(f"  {d.metadata['ticket_id']:12} | {d.metadata['category']:16} | {d.metadata['priority']}")

# (a) no filter — pure similarity
docs_none = vector_store.similarity_search(query, k=3)
show("NO FILTER:", docs_none)

# (b) category filter — CHANGE the value to a real one from the recon step
docs_cat = vector_store.similarity_search(query, k=3, filter={"category": "Authentication"})
show("CATEGORY = Authentication:", docs_cat)

# (c) priority filter — CHANGE the value to a real one from the recon step
docs_pri = vector_store.similarity_search(query, k=3, filter={"priority": "High"})
show("PRIORITY = High:", docs_pri)


NO FILTER:
  TICK-005     | Performance      | High
  TICK-012     | Reporting        | Medium
  TICK-007     | Search           | Critical

CATEGORY = Authentication:
  TICK-014     | Authentication   | Medium
  TICK-016     | Authentication   | High
  TICK-011     | Authentication   | Critical

PRIORITY = High:
  TICK-005     | Performance      | High
  TICK-003     | Payment          | High
  TICK-016     | Authentication   | High


In [16]:
streaming_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | PROMPT
    | llm            # your normal llm — no streaming setup needed
    | StrOutputParser()
)

query = "How do I fix database timeouts?"
print("Streaming answer:\n")
for chunk in streaming_chain.stream(query):
    print(chunk, end="", flush=True)   # print each piece as it arrives

Streaming answer:

To fix database timeouts, you can refer to the resolution provided in Ticket ID: TICK-002. The issue was caused by the database connection pool being sized too small for peak load, which led to 'connection pool exhausted' and 'timeout waiting for connection' errors. The resolution involved the following steps:

1. Increased the `max_connections` from 20 to 100 to accommodate higher demand.
2. Added connection pooling monitoring and alerts to keep track of connection usage.
3. Optimized long-running queries that were holding connections.

Implementing these changes should help mitigate database timeouts.

In [17]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_history = []

conv_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using context: {context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

def ask_with_history(question):
    context = format_docs(retriever.invoke(question))
    chain = conv_prompt | llm | StrOutputParser()
    response = chain.invoke({"context": context, "history": chat_history, "question": question})
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response))
    return response

print("Q1: What causes authentication failures?")
print(ask_with_history("What causes authentication failures?"))
print("\n" + "="*60 + "\n")
print("Q2: How do I fix it?")
print(ask_with_history("How do I fix it?"))

Q1: What causes authentication failures?
Authentication failures can be caused by a variety of issues, including:

1. **Incorrect Credentials**: Users may enter the wrong username or password, leading to authentication failures.

2. **Password Reset Issues**: As seen in Ticket TICK-001, if the password hash algorithm is updated without invalidating existing session tokens, users may face authentication failures after a password reset.

3. **Two-Factor Authentication Problems**: Issues with time-sensitive codes, such as those generated by authenticator apps, can lead to failures. For example, in Ticket TICK-016, server time drift caused TOTP codes to be invalid.

4. **Configuration Errors**: Misconfigurations in authentication systems, such as SSO setups, can lead to failures. In Ticket TICK-011, an upgrade changed the default SAML signature algorithm, which was not reflected in the Identity Provider (IdP) settings.

5. **Session Management Issues**: If session tokens are not properly m